# AiXbio — Notebook 2: Probes + Activation Steering

Replaces LLC (requires careful hyperparameter calibration for non-LM probes).

**Experiment 2** — Layer sweep: which ESM-2 layer best separates toxins from controls?

**Experiment 3** — Activation steering: is toxicity *causally* encoded as a linear direction?
   - Probe steering: add probe weight vector to control embeddings
   - SAE feature steering: add top SAE decoder directions to control embeddings

**Generalization curve** — does the steering direction transfer to redesigned sequences?

In [ ]:
import warnings, json
from pathlib import Path
import numpy as np
import torch, torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
LAYERS = [1, 9, 18, 24, 30, 33]
Path('results').mkdir(exist_ok=True)
print(f'Device: {DEVICE}')

## Probe

In [ ]:
class ToxinProbe(nn.Module):
    def __init__(self, d=1280):
        super().__init__()
        self.linear = nn.Linear(d, 1)
    def forward(self, x):
        return self.linear(x).squeeze(-1)

def train_probe(X, y, lr=1e-2, epochs=200, batch_size=256):
    probe = ToxinProbe(X.shape[1]).to(DEVICE)
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32),
                        torch.tensor(y, dtype=torch.float32))
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)
    crit = nn.BCEWithLogitsLoss()
    opt  = torch.optim.Adam(probe.parameters(), lr=lr, weight_decay=1e-4)
    probe.train()
    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); crit(probe(xb), yb).backward(); opt.step()
    probe.eval()
    with torch.no_grad():
        loss_star = crit(
            probe(torch.tensor(X, dtype=torch.float32).to(DEVICE)),
            torch.tensor(y, dtype=torch.float32).to(DEVICE)
        ).item()
    return probe, loss_star

print('ToxinProbe defined.')

## Experiment 1 — Layer Sweep

In [ ]:
layer_results = {}; BEST_LAYER = 24; best_auroc = 0

for layer in LAYERS:
    tox  = np.load(f'embeddings/natural_toxins_layer{layer}.npy')
    ctrl = np.load(f'embeddings/controls_layer{layer}.npy')
    X = np.concatenate([tox, ctrl])
    y = np.concatenate([np.ones(len(tox)), np.zeros(len(ctrl))])
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42)
    probe, loss_star = train_probe(X_tr, y_tr)
    with torch.no_grad():
        probs = torch.sigmoid(
            probe(torch.tensor(X_te, dtype=torch.float32).to(DEVICE))
        ).cpu().numpy()
    auroc = roc_auc_score(y_te, probs)
    layer_results[layer] = {'auroc': float(auroc), 'loss_star': float(loss_star)}
    print(f'  Layer {layer:2d}: AUROC={auroc:.4f}')
    if auroc > best_auroc:
        best_auroc = auroc; BEST_LAYER = layer

print(f'\nBest layer: {BEST_LAYER} (AUROC={best_auroc:.4f})')
with open('results/layer_sweep.json', 'w') as f:
    json.dump({'layer_results': {str(k):v for k,v in layer_results.items()},
               'best_layer': BEST_LAYER}, f, indent=2)

## Experiment 3 — Activation Steering

We ask: is toxicity a *causal* linear direction in ESM-2's representation space?

**Probe steering**: the trained linear probe defines a direction `w` in embedding space.
Adding `α·w` to a control embedding should smoothly increase its toxin score.

If the probe is reading a real causal feature, steering should transfer to *out-of-distribution* sequences (redesigns at low identity). If it's a surface correlate, it won't.

In [ ]:
# Load best-layer embeddings
tox_embs  = np.load(f'embeddings/natural_toxins_layer{BEST_LAYER}.npy')
ctrl_embs = np.load(f'embeddings/controls_layer{BEST_LAYER}.npy')
rdsg_embs = np.load(f'embeddings/redesigns_layer{BEST_LAYER}.npy')
seq_identities   = np.load('embeddings/sequence_identities.npy')
blast_identities = np.load('embeddings/blast_identities.npy')

X_all = np.concatenate([tox_embs, ctrl_embs])
y_all = np.concatenate([np.ones(len(tox_embs)), np.zeros(len(ctrl_embs))])
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=42)

probe, loss_star = train_probe(X_train, y_train)

# ── Steering vector: normalised probe weight ──────────────────
# The probe's linear weight IS the toxin direction in embedding space
w = probe.linear.weight.data.cpu().numpy().squeeze()   # (d,)
w_norm = w / np.linalg.norm(w)                         # unit vector

# Mean-difference vector as alternative
tox_mean  = tox_embs.mean(axis=0)
ctrl_mean = ctrl_embs.mean(axis=0)
diff_vec  = tox_mean - ctrl_mean
diff_norm = diff_vec / np.linalg.norm(diff_vec)

print(f'Probe weight direction: ||w|| = {np.linalg.norm(w):.4f}')
print(f'Mean-diff direction:    ||Δμ|| = {np.linalg.norm(diff_vec):.4f}')
print(f'Cosine(w, Δμ) = {np.dot(w_norm, diff_norm):.4f}')
print(f'  (> 0.9 → probe learned the mean-difference direction)')

In [ ]:
def probe_score(embs):
    """Sigmoid(probe(x)) → toxin probability."""
    with torch.no_grad():
        return torch.sigmoid(
            probe(torch.tensor(embs, dtype=torch.float32).to(DEVICE))
        ).cpu().numpy()


def steer_and_score(embs, direction, alphas):
    """
    For each alpha, add alpha * direction to embeddings and score.
    Returns (n_alphas, n_seqs) score matrix.
    """
    scores = []
    for alpha in alphas:
        steered = embs + alpha * direction
        scores.append(probe_score(steered))
    return np.stack(scores)  # (n_alphas, n_seqs)


# Steering range: span ±5 standard deviations of the probe score distribution
emb_std = np.std(X_train @ w_norm)
alphas  = np.linspace(-3*emb_std, 3*emb_std, 31)

# Steer controls along probe direction
ctrl_test = X_test[y_test == 0]
ctrl_baseline_score = probe_score(ctrl_test).mean()
ctrl_steered_scores = steer_and_score(ctrl_test, w_norm, alphas).mean(axis=1)

print('Probe steering on control sequences:')
print(f'  Baseline score: {ctrl_baseline_score:.4f}')
print(f'  Score at α=+3σ: {ctrl_steered_scores[-1]:.4f}')
print(f'  Score at α=-3σ: {ctrl_steered_scores[0]:.4f}')

# Steer redesigns
rdsg_baseline_score = probe_score(rdsg_embs).mean()
rdsg_steered_scores = steer_and_score(rdsg_embs, w_norm, alphas).mean(axis=1)
print(f'\nRedesign baseline score: {rdsg_baseline_score:.4f}')
print(f'Tox baseline score:      {probe_score(tox_embs).mean():.4f}')

# Save steering curve
steering_results = {
    'alphas':  alphas.tolist(),
    'ctrl_steered_scores':  ctrl_steered_scores.tolist(),
    'rdsg_steered_scores':  rdsg_steered_scores.tolist(),
    'ctrl_baseline_score':  float(ctrl_baseline_score),
    'rdsg_baseline_score':  float(rdsg_baseline_score),
    'emb_std': float(emb_std),
    'cosine_w_diff': float(np.dot(w_norm, diff_norm)),
}

## Experiment 3b — Does Steering Transfer Across Identity Bins?

This is the key biosecurity claim: if we steer a low-identity redesign toward the toxin direction,
does the probe score it as toxic? If yes → the feature is mechanistically robust.

In [ ]:
IDENTITY_BINS = [
    (0.90,1.00,'90–100%'),(0.70,0.90,'70–90%'),(0.50,0.70,'50–70%'),
    (0.30,0.50,'30–50%'),(0.10,0.30,'10–30%'),(0.00,0.10,'<10%'),
]

# Full test set: natural test + redesigns
X_test_all  = np.concatenate([X_test, rdsg_embs])
y_test_all  = np.concatenate([y_test, np.ones(len(rdsg_embs))])

# Steering multiplier that pushes average control to 0.5 (decision boundary)
# We use a fixed alpha = 2σ for comparison
alpha_fixed = 2 * emb_std

def blast_sens(mask, thr=0.30):
    if mask.sum() < 5: return float('nan')
    y = y_test_all[mask]; pred = (blast_identities[mask] >= thr).astype(int)
    tp=((pred==1)&(y==1)).sum(); fn=((pred==0)&(y==1)).sum()
    return float(tp/(tp+fn)) if (tp+fn)>0 else float('nan')

gen_curve = []
print(f'{"Bin":10s}  {"n":>5}  {"Probe":>8}  {"Steered":>8}  {"BLAST":>8}')
print('-'*50)
for lo, hi, label in IDENTITY_BINS:
    mask = (seq_identities >= lo) & (seq_identities < hi)
    n = int(mask.sum())
    if n < 5:
        print(f'{label:10s}  {n:>5}  {"–":>8}  {"–":>8}  {"–":>8}')
        continue

    # Natural probe AUROC
    X_bin = X_test_all[mask]; y_bin = y_test_all[mask]
    probs = probe_score(X_bin)
    auroc_nat = roc_auc_score(y_bin, probs) if len(np.unique(y_bin)) > 1 else float('nan')

    # Steered: add α·w to every sequence in bin, re-score
    probs_steered = probe_score(X_bin + alpha_fixed * w_norm)
    auroc_steered = roc_auc_score(y_bin, probs_steered) if len(np.unique(y_bin)) > 1 else float('nan')

    bl = blast_sens(mask)
    gen_curve.append({'bin': label, 'lo': lo, 'hi': hi, 'n': n,
                       'probe_auroc': float(auroc_nat),
                       'steered_auroc': float(auroc_steered),
                       'blast_sensitivity': float(bl),
                       'alpha_fixed': float(alpha_fixed)})
    print(f'{label:10s}  {n:>5}  {auroc_nat:>8.3f}  {auroc_steered:>8.3f}  {bl:>8.3f}')

print(f'\nSteered AUROC > probe AUROC at low identity?  '
      f'{"YES" if any(e.get("steered_auroc",0)>e.get("probe_auroc",0) for e in gen_curve if e.get("lo",1)<0.3) else "NO"}')
print('  YES → steering amplifies a real causal signal')
print('  NO  → probe saturated; redesigns already score high without steering')

In [ ]:
# Save all results
def _conv(o):
    if isinstance(o,(np.integer,np.floating)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return o

with torch.no_grad():
    probs_nat = probe_score(X_test)
auroc_nat = roc_auc_score(y_test, probs_nat)

all_results = {
    'best_layer': BEST_LAYER,
    'auroc_natural': float(auroc_nat),
    'steering': steering_results,
    'gen_curve': gen_curve,
    'layer_sweep': {str(k): v for k, v in layer_results.items()},
    'method': 'activation_steering',
    'notes': 'LLC replaced by activation steering (v2 API incompatible with probes)',
}
with open('results/main_results.json', 'w') as f:
    json.dump(all_results, f, default=_conv, indent=2)
print('Saved → results/main_results.json')
print(f'\nAUROC (natural test): {auroc_nat:.4f}')
print(f'Best layer:           {BEST_LAYER}')
print(f'Steering effect:      ctrl score {ctrl_baseline_score:.3f} → {ctrl_steered_scores[-1]:.3f} at α=+3σ')